# Notebook 07 — Decorators & generators

| | |
|---|---|
| **Track** | AI Engineering · `03-python-foundations/exercises/` |
| **Level** | Advanced |
| **Pattern** | *Concept* → *Runnable demo* → *Your turn* → *Solution* |

**Prerequisites:** `06-exceptions-context-managers.ipynb`

**Next up:** `08-advanced-typing-protocols-dataclasses.ipynb`

---

## Learning objectives

- Implement decorators with `functools.wraps`.
- Write memory-efficient iterators with `yield`.
- Compose timing/logging cross-cutting concerns.

## Table of contents

1. **Decorator anatomy**
2. **Parameterized decorators**
3. **Generators for chunks**

---

## How to use this notebook

1. Run cells **top to bottom** the first time through.
2. Re-run and **change values** to test your understanding.
3. Do **Your turn** cells before opening solutions (HTML `<details>` blocks or later cells).

---


## 1 · Timing decorator

*Explanation:* Wrap functions without rewriting bodies.


In [ ]:
import functools
import time

def timed(fn):
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        t0 = time.perf_counter()
        out = fn(*args, **kwargs)
        print(f"{fn.__name__}: {(time.perf_counter()-t0)*1000:.1f} ms")
        return out
    return wrapper

@timed
def slow(n: int = 100_000) -> int:
    return sum(range(n))

slow()


## 2 · Chunk generator

*Explanation:* Yield windows — backbone of streaming ingestion.


In [ ]:
from collections.abc import Iterator

def char_windows(text: str, n: int, step: int) -> Iterator[str]:
    i = 0
    while i + n <= len(text):
        yield text[i : i + n]
        i += step

print(list(char_windows("abcdefghij", 4, 3)))


### Exercise — repeat decorator

`@repeat_times(n)` runs underlying function `n` times, returns **list** of results (assume no exceptions).


In [ ]:
import functools


def repeat_times(n: int):
    def deco(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            raise NotImplementedError

        return wrapper

    return deco


counter = {"v": 0}

@repeat_times(3)
def bump():
    counter["v"] += 1
    return counter["v"]

assert bump() == [1, 2, 3]
print("OK")


### Solution — repeat decorator

<details>
<summary>Click to expand</summary>

```python
import functools

def repeat_times(n: int):
    def deco(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            return [fn(*args, **kwargs) for _ in range(n)]
        return wrapper
    return deco
```

</details>
